# Model - Payment Regression | Linear Regression & MLP 
Created by Guillermo Arredondo Renero

Creation date: May 5, 2026  
Last updated: May 5, 2026

**Main objective**: Find and train best Linear Regression model. It is meant to be a baseline model. Then find and train a Multi Layer Perceptron

In [3]:
# ---------------------------- Libraries
## Directories
import os
## Data manipulation
import pandas as pd
import numpy as np

## Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable

# Extras
import warnings
warnings.filterwarnings('ignore')

# Modeling
from sklearn.linear_model import LinearRegression, ElasticNet
import tensorflow as tf
from tensorflow.keras import layers, models
from keras.models import Sequential
from keras.layers import Activation, BatchNormalization
from keras.layers import Dropout, Dense, Input
from keras.optimizers import Adam

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (root_mean_squared_error, 
                             mean_absolute_error, 
                             r2_score)
from scipy import stats

import optuna
from optuna.visualization import plot_optimization_history

import joblib
import pickle
import shap

RANDOM_STATE = 56

In [4]:
# ---------------------------- Directories
# Get main directory
CURRENT_DIR = os.getcwd()
MAIN_DIR = os.path.dirname(CURRENT_DIR)
PROCESSED_DIR = os.path.join(MAIN_DIR, 'data', 'processed', 'continuous_prediction')
MODELS_DIR = os.path.join(MAIN_DIR, 'outputs', 'models', 'continuous_prediction')
os.makedirs(MODELS_DIR, exist_ok=True)

## Data loading

In [5]:
# train = pd.read_csv(os.path.join(PROCESSED_DIR, 'train', 'train_data.csv'))
train_scaled =  pd.read_csv(os.path.join(PROCESSED_DIR, 'train', 'train_data_scaled.csv'))

# test = pd.read_csv(os.path.join(PROCESSED_DIR, 'test', 'test_data.csv'))
test_scaled =  pd.read_csv(os.path.join(PROCESSED_DIR, 'test', 'test_data_scaled.csv'))

# val = pd.read_csv(os.path.join(PROCESSED_DIR, 'val', 'val_data.csv'))
val_scaled =  pd.read_csv(os.path.join(PROCESSED_DIR, 'val', 'val_data_scaled.csv'))

In [6]:
# Quick sanity checks
print(f"Train set shape: {train_scaled.shape}")
print(f"Validation set shape: {val_scaled.shape}")
print(f"Test set shape: {test_scaled.shape}")

# assert not train.isnull().any().any(), "Train set contains null values"
# assert not val.isnull().any().any(), "Validation set contains null values"
# assert not test.isnull().any().any(), "Test set contains null values"

assert not train_scaled.isnull().any().any(), "Train scaled set contains null values"
assert not val_scaled.isnull().any().any(), "Validation scaled set contains null values"
assert not test_scaled.isnull().any().any(), "Test scaled set contains null values"

# print("Original info:")
# print(train.info())
# print(test.info())
# print(val.info())
# print("="*80)
print("Scaled info:")
print(train_scaled.info())
print(test_scaled.info())
print(val_scaled.info())

Train set shape: (19240, 24)
Validation set shape: (4440, 24)
Test set shape: (5921, 24)
Scaled info:
<class 'pandas.DataFrame'>
RangeIndex: 19240 entries, 0 to 19239
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   LIMIT_BAL              19240 non-null  float64
 1   AGE                    19240 non-null  float64
 2   BILL_AMT4              19240 non-null  float64
 3   BILL_AMT5              19240 non-null  float64
 4   BILL_AMT6              19240 non-null  float64
 5   PAY_AMT5               19240 non-null  float64
 6   PAY_AMT6               19240 non-null  float64
 7   credit_utilization     19240 non-null  float64
 8   avg_monthly_bill       19240 non-null  float64
 9   avg_monthly_payment    19240 non-null  float64
 10  SEX                    19240 non-null  float64
 11  PAY_5                  19240 non-null  float64
 12  PAY_6                  19240 non-null  float64
 13  Female         

In [8]:
# x_train = train.drop('PAY_AMT4', axis = 1)
# y_train = train['PAY_AMT4']

x_train_scaled = train_scaled.drop('PAY_AMT4', axis = 1)
y_train_scaled = train_scaled['PAY_AMT4']
#-----------
# x_test = test.drop('PAY_AMT4', axis = 1)
# y_test = test['PAY_AMT4']

x_test_scaled = test_scaled.drop('PAY_AMT4', axis = 1)
y_test_scaled = test_scaled['PAY_AMT4']
#-----------
# x_val = val.drop('PAY_AMT4', axis = 1)
# y_val = val['PAY_AMT4']

x_val_scaled = val_scaled.drop('PAY_AMT4', axis = 1)
y_val_scaled = val_scaled['PAY_AMT4']

Para este caso no utilizamos *FEATURE SELECTION* ya que las medidas de regularización que aplicaremos para cualquiera de los dos modelos manejan la importancia de las variables implícitamente.

## Model Training and Hyperparameter Tuning

### Linear Regression

In [9]:
# Get best Elastic Net model from Optuna study
# Using Elastic Net generalizes both Lasso and Ridge regression, allowing for a combination of L1 and L2 regularization. 
# This can help prevent overfitting while still performing feature selection, 
# making it a good choice for regression problems with many features.
def tune_elastic_net(X_train, y_train, X_val, y_val, n_trials, metric):
    def optuna_objective(trial):
        # Define hyperparameters to tune
        params = {
            'alpha' : trial.suggest_float('alpha', 0.0001, 1.0, log=True),
            'l1_ratio' : trial.suggest_float('l1_ratio', 0.0, 1.0),
            'fit_intercept' : trial.suggest_categorical('fit_intercept', [True, False]),
            'max_iter' : trial.suggest_int('max_iter', 1000, 10000),
        }        

        # Create model
        model = ElasticNet(**params, random_state=RANDOM_STATE)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)

        # Compute all metrics
        metrics = {
            'rmse':  root_mean_squared_error(y_val, y_pred),
            'mae': mean_absolute_error(y_val, y_pred),
            'r2': r2_score(y_val, y_pred)
        }
        
        # Log all metrics but return only the specified one for optimization
        for metric_name, metric_value in metrics.items():
            trial.set_user_attr(metric_name, metric_value)
        
        return metrics[metric]

    # Create Optuna study
    # Create Optuna study
    study = optuna.create_study(
        direction='minimize' if metric in ['rmse', 'mae'] else 'maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )

    # Optimize
    print("\nRunning Optuna optimization (this may take a few minutes)...")
    study.optimize(optuna_objective, n_trials=n_trials, n_jobs=-1, show_progress_bar=True)

    return study, study.best_params, study.best_value

In [10]:
lr_study, lr_best_params, lr_best_score = tune_elastic_net(
    x_train_scaled, y_train_scaled, x_val_scaled, y_val_scaled, n_trials=35, metric='rmse'
)

# Visualize optimization history
fig = plot_optimization_history(lr_study).show()
print(f"\n✅ Optimization Complete!")
print(f"Best AUC-ROC (CV): {lr_best_score:.4f}")
print(f"\nBest Hyperparameters:")
for param, value in lr_best_params.items():
    print(f"  • {param}: {value}")

# Save Optuna results
optuna_results_lr = {
    'lr_best_params': lr_best_params,
    'lr_best_score': lr_best_score,
    'n_trials': len(lr_study.trials),
    'trials_df': lr_study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs'))
}

optuna_results_lr['trials_df'].to_csv(os.path.join(MODELS_DIR, 'lr_optuna_trials.csv'), index=False)
optuna_results_lr['trials_df'].sort_values('value', ascending=False).head(10)

[I 2026-05-06 00:46:49,868] A new study created in memory with name: no-name-1b7fc55a-4d64-491b-9113-03040e3681b5



Running Optuna optimization (this may take a few minutes)...


Best trial: 3. Best value: 13086.7:   3%|▎         | 1/35 [00:03<01:55,  3.41s/it]

[I 2026-05-06 00:46:53,267] Trial 3 finished with value: 13086.680444571657 and parameters: {'alpha': 0.24633233893814932, 'l1_ratio': 0.42433159070559556, 'fit_intercept': True, 'max_iter': 4857}. Best is trial 3 with value: 13086.680444571657.


Best trial: 5. Best value: 12991.6:   6%|▌         | 2/35 [00:04<00:59,  1.80s/it]

...

[I 2026-05-06 00:50:17,416] Trial 26 finished with value: 12150.25730633025 and parameters: {'alpha': 0.0023643219861561645, 'l1_ratio': 0.3235413718401683, 'fit_intercept': True, 'max_iter': 6239}. Best is trial 22 with value: 12150.151503923811.


Best trial: 22. Best value: 12150.2: 100%|██████████| 35/35 [03:38<00:00,  6.24s/it]


[I 2026-05-06 00:50:28,420] Trial 34 finished with value: 12151.162127285139 and parameters: {'alpha': 0.00036904914084146456, 'l1_ratio': 0.3591887115753392, 'fit_intercept': True, 'max_iter': 3799}. Best is trial 22 with value: 12150.151503923811.



✅ Optimization Complete!
Best AUC-ROC (CV): 12150.1515

Best Hyperparameters:
  • alpha: 0.0021767274711916545
  • l1_ratio: 0.30361650137244023
  • fit_intercept: False
  • max_iter: 3247


,number,value,params_alpha,params_fit_intercept,params_l1_ratio,params_max_iter,user_attrs_mae,user_attrs_r2,user_attrs_rmse
10,10,13607.816927,0.520982,False,0.404698,4335,4516.580059,0.221986,13607.816927
3,3,13086.680445,0.246332,True,0.424332,4857,4744.073652,0.280436,13086.680445
5,5,12991.564522,0.556306,False,0.782290,9762,4706.831679,0.290857,12991.564522
11,11,12493.931226,0.056499,False,0.162824,5209,4861.335408,0.344143,12493.931226
9,9,12245.912417,0.020391,False,0.059868,7403,4974.485823,0.369924,12245.912417
0,0,12206.341145,0.052880,False,0.732815,5260,5004.269880,0.373989,12206.341145
15,15,12189.193172,0.041043,True,0.715292,9168,5027.972792,0.375747,12189.193172
33,33,12184.139923,0.016334,True,0.332935,3728,5032.970837,0.376265,12184.139923
8,8,12153.170447,0.004962,False,0.178549,5075,5079.571709,0.379431,12153.170447
7,7,12153.116033,0.008146,True,0.507723,4481,5083.704920,0.379437,12153.116033


In [11]:
joblib.dump(optuna_results_lr, os.path.join(MODELS_DIR, 'lr_optuna_results.pkl'))

['c:\\Users\\Memit\\OneDrive - INSTITUTO TECNOLOGICO AUTONOMO DE MEXICO\\Documentos\\AplicacionesTrabajo\\Bluetab\\default_credit_prediction\\outputs\\models\\continuous_prediction\\lr_optuna_results.pkl']

### MLP with DropOut

In [12]:
def mlp_model(input_dim, params):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    
    # Add hidden layers with specified parameters
    for i in range(params['num_hidden_layers']):
        model.add(Dense(params['units_per_layer'], activation=params['activation']))
        if params['dropout_rate'] > 0:
            model.add(Dropout(params['dropout_rate']))
    
    # Output layer for regression
    model.add(Dense(1, activation='linear'))
    
    # Compile the model
    optimizer = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=optimizer, loss='mean_squared_error', 
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), 
                           tf.keras.metrics.MeanAbsoluteError(),
                           tf.keras.metrics.R2Score()])
    
    return model

In [ ]:
# Try different hyperparameters for the MLP model using optuna
def mlp_objective(trial):
    params ={
        'num_hidden_layers': trial.suggest_int('num_hidden_layers', 1, 5),
        'units_per_layer': trial.suggest_categorical('units_per_layer', [16, 32, 64, 128]),
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.0, 0.5),
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
    }
    model = mlp_model(input_dim=x_train_scaled.shape[1], params=params)

    model.fit(x_train_scaled, y_train_scaled, 
              validation_data=(x_val_scaled, y_val_scaled), epochs = 100, 
              batch_size=32, verbose=0)
    
    val_loss, val_rmse, val_mae, val_r2 = model.evaluate(x_val_scaled, y_val_scaled, verbose=0)

    trial.set_user_attr('val_loss', val_loss)
    trial.set_user_attr('val_rmse', val_rmse)
    trial.set_user_attr('val_mae', val_mae)
    trial.set_user_attr('val_r2', val_r2)
    return val_rmse

mlp_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))

# Optimize
print("\nRunning Optuna optimization (this may take a few minutes)...")
mlp_study.optimize(mlp_objective, n_trials=35, n_jobs=-1, show_progress_bar=True)

[I 2026-05-06 00:50:29,684] A new study created in memory with name: no-name-c8bd734b-dc6b-4d9a-a701-17b0881cbf10



Running Optuna optimization (this may take a few minutes)...


  0%|          | 0/35 [00:00<?, ?it/s]

In [ ]:
print(f"\n✅ Optimization Complete!")
print(f"Best RMSE: {mlp_study.best_value:.4f}")
print(f"\nBest Hyperparameters:")
for param, value in mlp_study.best_params.items():
    print(f"  • {param}: {value}")

# Visualize optimization history
fig = plot_optimization_history(mlp_study).show()

# Save Optuna results
optuna_results_mlp = {
    'mlp_best_params': mlp_study.best_params,
    'mlp_best_score': mlp_study.best_value,
    'n_trials': len(mlp_study.trials),
    'trials_df': mlp_study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs'))
}

optuna_results_mlp['trials_df'].to_csv(os.path.join(MODELS_DIR, 'mlp_optuna_trials.csv'), index=False)
optuna_results_mlp['trials_df'].sort_values('value', ascending=False).head(10)

In [ ]:
joblib.dump(optuna_results_mlp, os.path.join(MODELS_DIR, 'mlp_optuna_results.pkl'))

## Predict & Evaluation

In [ ]:
print("FINAL MODEL TRAINING & EVALUATION | Linear Regression (ElasticNet)")
print("="*80)

# Use best model
print("\nTraining final model with best hyperparameters...")
best_lr_model = ElasticNet(
    alpha=lr_best_params['alpha'],
    l1_ratio = lr_best_params['l1_ratio'],
    fit_intercept=lr_best_params['fit_intercept'],
    max_iter=lr_best_params['max_iter'],
    random_state=RANDOM_STATE
)

# Train on full training set
best_lr_model.fit(x_train_scaled, y_train_scaled)

# ========== VALIDATION SET EVALUATION ==========
print("\n### Validation Set Performance")
print("-" * 80)

y_tr_pred = best_lr_model.predict(x_train_scaled)
y_val_pred = best_lr_model.predict(x_val_scaled)

val_metrics = {
    'rmse': root_mean_squared_error(y_val_scaled, y_val_pred),
    'mae': mean_absolute_error(y_val_scaled, y_val_pred),
    'r2': r2_score(y_val_scaled, y_val_pred)
}

for metric, value in val_metrics.items():
    print(f"{metric:15s}: {value:.4f}")

# ========== TEST SET EVALUATION ==========
print("\n### Test Set Performance")
print("-" * 80)

y_test_pred = best_lr_model.predict(x_test_scaled)

test_metrics = {
    'rmse': root_mean_squared_error(y_test_scaled, y_test_pred),
    'mae': mean_absolute_error(y_test_scaled, y_test_pred),
    'r2': r2_score(y_test_scaled, y_test_pred)
}

for metric, value in test_metrics.items():
    print(f"{metric:15s}: {value:.4f}")

# =========== Predicted vs Actual Plot ===========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(x=y_val_scaled, y=y_val_pred, alpha=0.5, ax=axes[0])
axes[0].plot(
    [y_val_scaled.min(), y_val_scaled.max()],
    [y_val_scaled.min(), y_val_scaled.max()],
    'r--'
)
axes[0].set_xlabel('Actual PAY_AMT4')
axes[0].set_ylabel('Predicted PAY_AMT4')
axes[0].set_title('Predicted vs Actual PAY_AMT4 (Validation Set)')

sns.scatterplot(x=y_test_scaled, y=y_test_pred, alpha=0.5, ax=axes[1])
axes[1].plot(
    [y_test_scaled.min(), y_test_scaled.max()],
    [y_test_scaled.min(), y_test_scaled.max()],
    'r--'
)
axes[1].set_xlabel('Actual PAY_AMT4')
axes[1].set_ylabel('Predicted PAY_AMT4')
axes[1].set_title('Predicted vs Actual PAY_AMT4 (Test Set)')

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'lr_predicted_vs_actual.png'), dpi=300, bbox_inches='tight')
plt.show()

# ========= Residuals Plot ==========
residuals_val = y_val_scaled - y_val_pred
residuals_test = y_test_scaled - y_test_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Predicted for Validation
sns.scatterplot(x=y_val_pred, y=residuals_val, alpha=0.5, ax=axes[0, 0])
axes[0, 0].axhline(0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Predicted PAY_AMT4')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted PAY_AMT4 (Validation Set)')

# Residuals vs Predicted for Test
sns.scatterplot(x=y_test_pred, y=residuals_test, alpha=0.5, ax=axes[0, 1])
axes[0, 1].axhline(0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Predicted PAY_AMT4')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals vs Predicted PAY_AMT4 (Test Set)')

# Q-Q plot for Validation residuals
stats.probplot(residuals_val, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot of Residuals (Validation Set)')

# Q-Q plot for Test residuals
stats.probplot(residuals_test, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Residuals (Test Set)')

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'lr_residuals_and_qq.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
print("\n### Feature Importance (Linear Regression Coefficients)")
print("-" * 80)

features = x_train_scaled.columns.tolist()

coefficients = pd.DataFrame({
    'feature': features,
    'coefficient': best_lr_model.coef_,
    'abs_coefficient': np.abs(best_lr_model.coef_)
}).sort_values('abs_coefficient', ascending=False)

retained  = coefficients[coefficients['coefficient'] != 0]
eliminated = coefficients[coefficients['coefficient'] == 0]

print(f"Features retained:   {len(retained)}")
print(f"Features eliminated: {len(eliminated)}")

print("\nTop 15 Most Important Features:")
print(coefficients.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
top_coefs = coefficients.head(20)
colors = ['#e74c3c' if x > 0 else '#3498db' for x in top_coefs['coefficient']]
ax.barh(range(len(top_coefs)), top_coefs['coefficient'].values, color=colors, alpha=0.7)
ax.set_yticks(range(len(top_coefs)))
ax.set_yticklabels(top_coefs['feature'].values, fontsize=10)
ax.set_xlabel('Coefficient Value', fontsize=11, fontweight='bold')
ax.set_title('Top 20 Feature Coefficients (ElasticNet Regression)', fontsize=12, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'elasticnet_feature_coefficients.png'), dpi=300, bbox_inches='tight')
plt.show()

print("\n### Saving Model and Artifacts")
print("-" * 80)

joblib.dump(best_lr_model, os.path.join(MODELS_DIR, 'elasticnet_model.pkl'))

with open(os.path.join(MODELS_DIR, 'elasticnet_retained_features.pkl'), 'wb') as f:
    pickle.dump(retained['feature'].tolist(), f)

results_summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'model_type': 'ElasticNet',
    'hyperparameters': lr_best_params,
    'retained_features': retained['feature'].tolist(),
    'n_features_selected': len(retained),
    'n_features_original': x_train_scaled.shape[1],
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
    'feature_importance': coefficients.to_dict(orient='list')
}

with open(os.path.join(MODELS_DIR, 'elasticnet_results_summary.pkl'), 'wb') as f:
    pickle.dump(results_summary, f)


### Feature Importance (Linear Regression Coefficients)
--------------------------------------------------------------------------------


In [ ]:
print("FINAL MODEL TRAINING & EVALUATION | MLP")
print("="*80)

# Use best model
print("\nTraining final model with best hyperparameters...")
best_mlp_model = mlp_model(
    input_dim=x_train_scaled.shape[1],
    params=mlp_study.best_params
)
# Train on full training set
history = best_mlp_model.fit(x_train_scaled, y_train_scaled,
                             validation_data=(x_val_scaled, y_val_scaled), epochs = 100, 
                          batch_size=32)

best_mlp_model.summary()

In [ ]:
# ========== ACCESS TRAINING HISTORY METRICS ==========
print("\n### Training History - Final Metrics")
print("=" * 80)

# Extract final epoch metrics
final_metrics = {
    'loss': history.history['loss'][-1],
    'val_loss': history.history['val_loss'][-1],
    'root_mean_squared_error': history.history['root_mean_squared_error'][-1],
    'val_root_mean_squared_error': history.history['val_root_mean_squared_error'][-1],
    'mean_absolute_error': history.history['mean_absolute_error'][-1],
    'val_mean_absolute_error': history.history['val_mean_absolute_error'][-1],
    'r2_score': history.history['r2_score'][-1],
    'val_r2_score': history.history['val_r2_score'][-1],
}

for metric, value in final_metrics.items():
    print(f"{metric:30s}: {value:.6f}")

# ========== PLOT TRAINING HISTORY ==========
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 0].set_title('Loss Over Epochs', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (MSE)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# RMSE
axes[0, 1].plot(history.history['root_mean_squared_error'], label='Training RMSE', linewidth=2)
axes[0, 1].plot(history.history['val_root_mean_squared_error'], label='Validation RMSE', linewidth=2)
axes[0, 1].set_title('RMSE Over Epochs', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('RMSE')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# MAE
axes[1, 0].plot(history.history['mean_absolute_error'], label='Training MAE', linewidth=2)
axes[1, 0].plot(history.history['val_mean_absolute_error'], label='Validation MAE', linewidth=2)
axes[1, 0].set_title('MAE Over Epochs', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('MAE')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# R² Score
axes[1, 1].plot(history.history['r2_score'], label='Training R²', linewidth=2)
axes[1, 1].plot(history.history['val_r2_score'], label='Validation R²', linewidth=2)
axes[1, 1].set_title('R² Score Over Epochs', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('R² Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'mlp_training_history.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ========== VALIDATION SET EVALUATION ==========
print("\n### Validation Set Performance")
print("-" * 80)

y_tr_pred = best_mlp_model.predict(x_train_scaled, verbose=0)
y_val_pred = best_mlp_model.predict(x_val_scaled, verbose=0)

val_metrics = {
    'rmse': root_mean_squared_error(y_val_scaled, y_val_pred),
    'mae': mean_absolute_error(y_val_scaled, y_val_pred),
    'r2': r2_score(y_val_scaled, y_val_pred)
}

for metric, value in val_metrics.items():
    print(f"{metric:15s}: {value:.4f}")

# ========== TEST SET EVALUATION ==========
print("\n### Test Set Performance")
print("-" * 80)

y_test_pred = best_mlp_model.predict(x_test_scaled, verbose=0)

test_metrics = {
    'rmse': root_mean_squared_error(y_test_scaled, y_test_pred),
    'mae': mean_absolute_error(y_test_scaled, y_test_pred),
    'r2': r2_score(y_test_scaled, y_test_pred)
}

for metric, value in test_metrics.items():
    print(f"{metric:15s}: {value:.4f}")

# =========== Predicted vs Actual Plot ===========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(x=y_val_scaled, y=y_val_pred, alpha=0.5, ax=axes[0])
axes[0].plot(
    [y_val_scaled.min(), y_val_scaled.max()],
    [y_val_scaled.min(), y_val_scaled.max()],
    'r--'
)
axes[0].set_xlabel('Actual PAY_AMT4')
axes[0].set_ylabel('Predicted PAY_AMT4')
axes[0].set_title('Predicted vs Actual PAY_AMT4 (Validation Set)')

sns.scatterplot(x=y_test_scaled, y=y_test_pred, alpha=0.5, ax=axes[1])
axes[1].plot(
    [y_test_scaled.min(), y_test_scaled.max()],
    [y_test_scaled.min(), y_test_scaled.max()],
    'r--'
)
axes[1].set_xlabel('Actual PAY_AMT4')
axes[1].set_ylabel('Predicted PAY_AMT4')
axes[1].set_title('Predicted vs Actual PAY_AMT4 (Test Set)')

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'mlp_predicted_vs_actual.png'), dpi=300, bbox_inches='tight')
plt.show()

# ========= Residuals Plot ==========
residuals_val = y_val_scaled - y_val_pred
residuals_test = y_test_scaled - y_test_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Predicted for Validation
sns.scatterplot(x=y_val_pred, y=residuals_val, alpha=0.5, ax=axes[0, 0])
axes[0, 0].axhline(0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Predicted PAY_AMT4')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted PAY_AMT4 (Validation Set)')

# Residuals vs Predicted for Test
sns.scatterplot(x=y_test_pred, y=residuals_test, alpha=0.5, ax=axes[0, 1])
axes[0, 1].axhline(0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Predicted PAY_AMT4')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals vs Predicted PAY_AMT4 (Test Set)')

# Q-Q plot for Validation residuals
stats.probplot(residuals_val, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot of Residuals (Validation Set)')

# Q-Q plot for Test residuals
stats.probplot(residuals_test, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Residuals (Test Set)')

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'mlp_residuals_and_qq.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
print("\n### Feature Importance & Interpretability (MLP)")
print("-" * 80)

selected_features = x_train_scaled.columns.tolist()
background = x_train_scaled.sample(n=min(100, len(x_train_scaled)), random_state=RANDOM_STATE)

explainer = shap.DeepExplainer(best_mlp_model, background, feature_names=selected_features)
shap_values = explainer.shap_values(x_test_scaled)

shap.summary_plot(shap_values, x_test_scaled, feature_names=selected_features)
shap.summary_plot(shap_values, x_test_scaled, feature_names=selected_features,
                  plot_type='bar')

print("\n### Saving Model and Artifacts")
print("-" * 80)

joblib.dump(best_mlp_model, os.path.join(MODELS_DIR, 'mlp_model.pkl'))

shap_df = pd.DataFrame(shap_values.values, columns=selected_features)
feature_importance = shap_df.abs().mean().sort_values(ascending=False)

results_summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'model_type': 'MLP+DropOut',
    'hyperparameters': mlp_study.best_params,
    'selected_features': selected_features,
    'n_features': len(selected_features),
    'n_features_original': x_train_scaled.shape[1],
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
    'feature_importance': feature_importance.to_dict()
}

with open(os.path.join(MODELS_DIR, 'MLP_results_summary.pkl'), 'wb') as f:
    pickle.dump(results_summary, f)


## Model Comparison Summary

In [ ]:
# Function to compute all required metrics
def compute_metrics(y_true, y_pred):
    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

# Compute metrics for Logistic Regression
# Train
y_train_pred_lr = best_lr_model.predict(x_train_scaled)
y_train_pred_proba_lr = best_lr_model.predict_proba(x_train_scaled)[:, 1]
train_acc_lr, train_rec_lr, train_f1_lr, train_auc_lr, train_gini_lr, train_ks_lr = compute_metrics(y_train_scaled, y_train_pred_lr, y_train_pred_proba_lr)

# Validation
y_val_pred_lr = best_lr_model.predict(x_val_scaled)
y_val_pred_proba_lr = best_lr_model.predict_proba(x_val_scaled)[:, 1]
val_acc_lr, val_rec_lr, val_f1_lr, val_auc_lr, val_gini_lr, val_ks_lr = compute_metrics(y_val_scaled, y_val_pred_lr, y_val_pred_proba_lr)

# Test
y_test_pred_lr = best_lr_model.predict(x_test_scaled)
y_test_pred_proba_lr = best_lr_model.predict_proba(x_test_scaled)[:, 1]
test_acc_lr, test_rec_lr, test_f1_lr, test_auc_lr, test_gini_lr, test_ks_lr = compute_metrics(y_test_scaled, y_test_pred_lr, y_test_pred_proba_lr)

# Compute metrics for XGBoost (using existing predictions) - possible fix for future
# Train
train_acc_xgb, train_rec_xgb, train_f1_xgb, train_auc_xgb, train_gini_xgb, train_ks_xgb = compute_metrics(y_train, y_tr_pred, y_tr_pred_proba)

# Validation
val_acc_xgb, val_rec_xgb, val_f1_xgb, val_auc_xgb, val_gini_xgb, val_ks_xgb = compute_metrics(y_val, y_val_pred, y_val_pred_proba)

# Test
test_acc_xgb, test_rec_xgb, test_f1_xgb, test_auc_xgb, test_gini_xgb, test_ks_xgb = compute_metrics(y_test, y_test_pred, y_test_pred_proba)

# Create DataFrame
comparison_df = pd.DataFrame({
    'Model': ['Elastic Net Regression'] * 3 + ['MLP + DropOut'] * 3,
    'Dataset': ['Train', 'Validation', 'Test'] * 2,
    'RMSE' : 

})
# Set double level index for Model and Dataset
comparison_df.set_index(['Model', 'Dataset'], inplace=True)
# Print the DataFrame
comparison_df

In [ ]:
# Best model by metric in test set
test_comparison = comparison_df.xs('Test', level='Dataset')

for metric in test_comparison.columns:
    best_model = test_comparison[metric].idxmax()
    best_value = test_comparison.loc[best_model, metric]
    print(f"Best {metric}: {best_model} with value {best_value:.4f}")

Best Accuracy: XGBoost with value 0.8237
Best Recall: Logistic Regression with value 0.6374
Best F1-Score: Logistic Regression with value 0.5182
Best AUC-ROC: XGBoost with value 0.7755
Best Gini: XGBoost with value 0.5511
Best KS: XGBoost with value 0.4159
